# Credit Card Fraud Analysis: Where the Bank Loses Money and Which Rules Pay Off

**Business problem:** a bank wants to know how much fraud it has, where it comes from, and which simple rules would cut losses without flooding customers with false alerts.

**Approach:** pure data analysis (pandas, groupby, charts and rule testing). No machine learning.

**Business questions**
1. How big is the fraud problem?
2. Which fraud type costs the most?
3. Where does fraud concentrate (channel, merchant, country, security flags)?
4. When does fraud happen (hour, weekend, month)?
5. Which customers and accounts are most at risk?
6. Which simple rules save the most money after the cost of false alerts?

**Tools:** Python (pandas, matplotlib, seaborn) and Power BI.

## Load the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

# put the CSV in a data/ folder next to the notebooks/ folder (works on Windows, Mac and Linux)
paths=["../data/credit_card_fraud_detection_2026.csv","data/credit_card_fraud_detection_2026.csv","credit_card_fraud_detection_2026.csv"]
x=pd.read_csv(next(p for p in paths if os.path.exists(p)))
x.head()

In [ ]:
x.shape

In [ ]:
x.info()

In [ ]:
x.describe().T

## Data cleaning

In [ ]:
x.isnull().sum()[x.isnull().sum()>0]

In [ ]:
# which channels have no browser?
x[x['browser'].isna()]['transaction_channel'].value_counts().reset_index()

In [ ]:
# no browser for In-store and ATM (not an error), so label them instead of deleting rows
x['browser']=x['browser'].fillna('None (not online)')
x.isnull().sum().sum()

In [ ]:
x['transaction_id'].duplicated().sum()

In [ ]:
x['transaction_datetime']=pd.to_datetime(x['transaction_datetime'])
x['Month']=x['transaction_datetime'].dt.month
x['Month_Name']=x['transaction_datetime'].dt.month_name()
x[['transaction_datetime','Month','Month_Name']].head()

**Data cleaning summary**
- 100,000 rows, 43 columns, no duplicate transactions.
- `browser` had 41,709 blanks. They were structural (in-store and ATM have no browser), so I labelled them "None (not online)" instead of deleting rows.
- `transaction_datetime` was stored as text; I converted it to a datetime and added Month columns.

## Business Question 1: How big is the fraud problem?

In [ ]:
Total_Transactions=len(x)
Fraud_Transactions=x['is_fraud'].sum()
Fraud_Rate=x['is_fraud'].mean()*100
Total_Amount=x['transaction_amount'].sum()
Amount_Lost=x['fraud_amount_lost'].sum()
print("Total_Transactions:",Total_Transactions)
print("Fraud_Transactions:",Fraud_Transactions)
print("Fraud_Rate %:",round(Fraud_Rate,2))
print("Total_Amount:",round(Total_Amount,2))
print("Amount_Lost:",round(Amount_Lost,2))

In [ ]:
x['is_fraud'].value_counts().reset_index()

In [ ]:
fraud=x.loc[x['is_fraud']==1]
legit=x.loc[x['is_fraud']==0]
print("Avg fraud amount:",round(fraud['transaction_amount'].mean(),2))
print("Avg legit amount:",round(legit['transaction_amount'].mean(),2))

**Finding 1**
- **Evidence:** 2,055 of 100,000 transactions (2.05%) were fraud, costing 279,842 out of 12.69M processed. The average fraud transaction (169.93) is about 35% larger than a legitimate one (125.97).
- **Insight:** fraud is rare but costly per case, so checking every transaction would waste effort.
- **Recommendation:** target controls at the segments where fraud concentrates (next questions).

## Business Question 2: Which fraud type costs the most?

In [ ]:
Fraud_Type_Loss=fraud.groupby('fraud_type')['fraud_amount_lost'].sum().reset_index().sort_values('fraud_amount_lost',ascending=False)
Fraud_Type_Loss['pct']=(Fraud_Type_Loss['fraud_amount_lost']/Fraud_Type_Loss['fraud_amount_lost'].sum()*100).round(1)
Fraud_Type_Loss

In [ ]:
Fraud_Type_Count=fraud['fraud_type'].value_counts().reset_index()
Fraud_Type_Count

In [ ]:
plt.figure(figsize=(8,5))
bars=plt.bar(Fraud_Type_Loss['fraud_type'],Fraud_Type_Loss['fraud_amount_lost'],color='red',alpha=0.8)
plt.bar_label(bars,fmt='%.0f')
plt.xlabel('Fraud Type',size=10)
plt.xticks(rotation=45)
plt.ylabel('Amount Lost',size=10)
plt.ticklabel_format(style='plain',axis='y')
plt.title('Money Lost by Fraud Type',size=16)
plt.tight_layout()
plt.show()

**Finding 2**
- **Evidence:** Card-Not-Present fraud is 150,504 of the 279,842 lost (53.8%) and also the most frequent type (1,054 of 2,055 cases). Account-Takeover is 22.3%, Synthetic-Identity 12.5%, Card-Present 11.5%.
- **Insight:** more than half the money lost comes from payments where the physical card is not seen.
- **Recommendation:** strengthen online and phone payment checks first (for example one-time passwords or 3-D Secure).

## Business Question 3: Where does fraud concentrate?

In [ ]:
# fraud rate = fraud cases / all transactions in the group, so busy segments are compared fairly
overall=x['is_fraud'].mean()*100
overall

In [ ]:
channel_rate=(x.groupby('transaction_channel')['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'}).sort_values('Fraud_Rate',ascending=False)
channel_rate

In [ ]:
plt.figure(figsize=(8,5))
bars=plt.bar(channel_rate['transaction_channel'],channel_rate['Fraud_Rate'],color='red',alpha=0.8)
plt.bar_label(bars,fmt='%.2f')
plt.axhline(overall,color='black',ls='--',label='Overall')
plt.xlabel('Channel',size=10)
plt.ylabel('Fraud Rate %',size=10)
plt.title('Fraud Rate by Channel',size=16)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
category_rate=(x.groupby('merchant_category')['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'}).sort_values('Fraud_Rate',ascending=True)
plt.figure(figsize=(9,6))
plt.barh(category_rate['merchant_category'],category_rate['Fraud_Rate'],color='red',alpha=0.8)
plt.axvline(overall,color='black',ls='--',label='Overall')
plt.xlabel('Fraud Rate %',size=10)
plt.title('Fraud Rate by Merchant Category',size=16)
plt.legend()
plt.tight_layout()
plt.show()
category_rate.sort_values('Fraud_Rate',ascending=False).head()

In [ ]:
country_rate=x.groupby('merchant_country')['is_fraud'].agg(['mean','sum','count']).reset_index()
country_rate['mean']=(country_rate['mean']*100).round(2)
country_rate=country_rate.rename(columns={'mean':'Fraud_Rate','sum':'Frauds','count':'Transactions'}).sort_values('Fraud_Rate',ascending=False)
country_rate.head(10)

In [ ]:
# the four top countries are the "high-risk" countries: how much of the business and fraud do they hold?
hr=(x.groupby('high_risk_country')['is_fraud'].agg(['mean','sum','count']).reset_index())
hr['mean']=hr['mean']*100
hr=hr.rename(columns={'mean':'Fraud_Rate','sum':'Frauds','count':'Transactions'})
hr['pct_of_transactions']=(hr['Transactions']/len(x)*100).round(1)
hr['pct_of_frauds']=(hr['Frauds']/x['is_fraud'].sum()*100).round(1)
hr

In [ ]:
# security flags: international, card present, chip, PIN
for col in ['international_transaction','card_present','used_chip','used_pin']:
    print((x.groupby(col)['is_fraud'].mean()*100).round(2).reset_index(),'\n')

**Finding 3**
- **Evidence:** Online has a 3.18% fraud rate versus 0.80% in-store. Four countries (Romania 12.9%, Russia 11.4%, Nigeria 10.9%, China 9.9%) are only 4.9% of transactions but 26.5% of all fraud cases (545 of 2,055). Online Retail (3.32%) and Travel and Airlines (2.92%) are the riskiest categories. Transactions without chip (2.65% vs 0.75%) or without PIN (2.34% vs 0.77%) are about 3 times riskier.
- **Insight:** fraud clusters in online payments and in four specific countries, and in-person chip and PIN payments are much safer.
- **Recommendation:** add step-up verification for online and international purchases, and apply extra checks to the four high-risk countries rather than blocking them outright.

## Business Question 4: When does fraud happen?

In [ ]:
hour_rate=(x.groupby('transaction_hour')['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'})
plt.figure(figsize=(10,5))
plt.bar(hour_rate['transaction_hour'],hour_rate['Fraud_Rate'],color='red',alpha=0.8)
plt.axhline(overall,color='black',ls='--',label='Overall')
plt.xlabel('Hour of Day',size=10)
plt.xticks(range(24))
plt.ylabel('Fraud Rate %',size=10)
plt.title('Fraud Rate by Hour',size=16)
plt.legend()
plt.tight_layout()
plt.show()
hour_rate.sort_values('Fraud_Rate',ascending=False).head()

In [ ]:
night=x.loc[x['transaction_hour'].between(0,4)]
print("Overnight transactions:",len(night))
print("Overnight frauds:",night['is_fraud'].sum())
print("Share of all fraud cases %:",round(night['is_fraud'].sum()/x['is_fraud'].sum()*100,1))

In [ ]:
x.groupby('is_weekend')['is_fraud'].mean().mul(100).round(2).reset_index()

In [ ]:
month_rate=(x.groupby(['Month','Month_Name'])['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'}).sort_values('Month')
plt.figure(figsize=(10,5))
plt.plot(month_rate['Month_Name'],month_rate['Fraud_Rate'],color='red',marker='o',lw=3)
plt.axhline(overall,color='black',ls='--',label='Overall')
plt.ylim(0,3)   # start at 0 so small changes are not exaggerated
plt.xlabel('Months',size=10)
plt.xticks(rotation=45)
plt.ylabel('Fraud Rate %',size=10)
plt.title('Monthly Fraud Rate',size=16)
plt.legend()
plt.tight_layout()
plt.show()

**Finding 4**
- **Evidence:** between 1am and 4am the fraud rate is 7.5% to 11.3% (against 2.05% overall), but those hours hold only 2,456 transactions and 150 frauds (7.3% of all fraud). Weekend and weekday are almost the same (2.09% vs 2.04%). Monthly rates stay between 1.87% and 2.29% with no clear trend.
- **Insight:** overnight is risky per transaction but small in volume, and there is no weekend or seasonal effect.
- **Recommendation:** add overnight monitoring as a secondary control, and do not build weekend or seasonal rules.

## Business Question 5: Which customers and accounts are most at risk?

In [ ]:
x['age_band']=pd.cut(x['customer_age'],[0,24,34,44,54,64,120],labels=['<25','25-34','35-44','45-54','55-64','65+'])
age_rate=(x.groupby('age_band',observed=True)['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'})
age_rate

In [ ]:
x['account_age_band']=pd.cut(x['account_age_days'],[-1,90,365,1095,1e9],labels=['<3 months','3-12 months','1-3 years','3+ years'])
acc=x.groupby('account_age_band',observed=True)['is_fraud'].agg(['mean','sum','count']).reset_index()
acc['mean']=acc['mean']*100
acc=acc.rename(columns={'mean':'Fraud_Rate','sum':'Frauds','count':'Transactions'})
acc

In [ ]:
plt.figure(figsize=(8,5))
bars=plt.bar(acc['account_age_band'].astype(str),acc['Fraud_Rate'],color='red',alpha=0.8)
plt.bar_label(bars,fmt='%.2f')
plt.axhline(overall,color='black',ls='--',label='Overall')
plt.xlabel('Account Age',size=10)
plt.ylabel('Fraud Rate %',size=10)
plt.title('Fraud Rate by Account Age',size=16)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cb=x.groupby('num_chargebacks_ever')['is_fraud'].agg(['mean','sum','count']).reset_index()
cb['mean']=cb['mean']*100
cb=cb.rename(columns={'mean':'Fraud_Rate','sum':'Frauds','count':'Transactions'})
cb

In [ ]:
state_rate=x.groupby('customer_state')['is_fraud'].agg(['mean','sum','count']).reset_index()
state_rate['mean']=state_rate['mean']*100
state_rate.rename(columns={'mean':'Fraud_Rate','sum':'Frauds','count':'Transactions'}).sort_values('Fraud_Rate',ascending=False).head(10)

In [ ]:
(x.groupby('email_domain')['is_fraud'].mean()*100).reset_index().rename(columns={'is_fraud':'Fraud_Rate'}).sort_values('Fraud_Rate',ascending=False)

**Finding 5**
- **Evidence:** accounts under 3 months old have a 4.85% fraud rate (515 frauds, 25% of all fraud from 10.6% of transactions) versus 1.46% for accounts older than 3 years. Customers with 2 or more past chargebacks run at 4.3% to 4.8%, against 1.76% for none. Customer age (1.9% to 2.2%), email domain (1.9% to 2.2%) and state (up to 2.9%) show no meaningful difference.
- **Insight:** risk follows account history, not customer demographics.
- **Recommendation:** apply extra verification to new accounts and repeat-chargeback customers, and do not target customers by age, email or state.

## Business Question 6: Which simple rules save the most money?

A rule is a plain "if" condition. Two measures judge a rule:
- **Recall** = share of all real frauds the rule catches.
- **Precision** = share of flagged transactions that were really fraud.

In [ ]:
rules={
 'High-risk country': x['high_risk_country']==1,
 'Overnight (0-4am)': x['transaction_hour'].between(0,4),
 'New account (<=90 days)': x['account_age_days']<=90,
 '2+ chargebacks': x['num_chargebacks_ever']>=2,
 'International + card not present': (x['international_transaction']==1)&(x['card_present']==0),
 'Amount > 3x customer average': x['amount_vs_avg_ratio']>3,
}

In [ ]:
total_fraud=x['is_fraud'].sum()
rows=[]
for name,mask in rules.items():
    flagged=int(mask.sum())
    caught=int(x.loc[mask,'is_fraud'].sum())
    rows.append({'Rule':name,'Flagged':flagged,'Frauds_Caught':caught,
                 'Recall_%':round(caught/total_fraud*100,1),
                 'Precision_%':round(caught/flagged*100,1),
                 'False_Alerts':flagged-caught})
rule_table=pd.DataFrame(rows)
rule_table

In [ ]:
# combine four rules: flag if ANY is true  (| means OR, & means AND)
combo=rules['High-risk country']|rules['New account (<=90 days)']|rules['2+ chargebacks']|rules['Overnight (0-4am)']
flagged=int(combo.sum())
caught=int(x.loc[combo,'is_fraud'].sum())
print("Flagged:",flagged)
print("Frauds caught:",caught)
print("Recall %:",round(caught/total_fraud*100,1))
print("Precision %:",round(caught/flagged*100,1))

### Rule savings after the cost of false alerts
- **Money saved** = fraud amount lost on the frauds the rule flags.
- **Cost of false alerts** = false alerts x cost per alert (analyst time and customer friction).
- **Net benefit** = money saved - cost of false alerts.

*Assumptions: a flagged fraud is prevented in full, and each false alert costs a flat 5. Change `COST_PER_FALSE_ALERT` and re-run.*

In [ ]:
COST_PER_FALSE_ALERT=5
rules['Combined (4 rules)']=combo

rows=[]
for name,mask in rules.items():
    saved=x.loc[mask,'fraud_amount_lost'].sum()
    false_alerts=int((mask&(x['is_fraud']==0)).sum())
    alert_cost=false_alerts*COST_PER_FALSE_ALERT
    rows.append({'Rule':name,'Money_Saved':round(saved),'False_Alerts':false_alerts,
                 'Alert_Cost':alert_cost,'Net_Benefit':round(saved-alert_cost)})
savings=pd.DataFrame(rows).sort_values('Net_Benefit',ascending=False)
savings

In [ ]:
plt.figure(figsize=(9,5))
colors=['green' if v>0 else 'red' for v in savings['Net_Benefit']]
bars=plt.barh(savings['Rule'][::-1],savings['Net_Benefit'][::-1],color=colors[::-1],alpha=0.8)
plt.bar_label(bars,fmt='%.0f')
plt.xlabel('Net Benefit',size=10)
plt.title('Net Benefit by Rule (cost per false alert = 5)',size=14)
plt.tight_layout()
plt.show()

In [ ]:
# how sensitive is each rule to the cost of a false alert?
sens=[]
for cost in [1,5,10]:
    for name,mask in rules.items():
        saved=x.loc[mask,'fraud_amount_lost'].sum()
        fp=int((mask&(x['is_fraud']==0)).sum())
        sens.append({'Cost_Per_Alert':cost,'Rule':name,'Net_Benefit':round(saved-fp*cost)})
sens=pd.DataFrame(sens)
sens.pivot(index='Rule',columns='Cost_Per_Alert',values='Net_Benefit')

### A rule-based risk score (no machine learning)
Instead of using one rule at a time, count **how many rules each transaction triggers**. A transaction that triggers several warning signs at once should be riskier than one that triggers one.

In [ ]:
rule_cols={'High-risk country':'flag_high_risk_country','Overnight (0-4am)':'flag_overnight',
           'New account (<=90 days)':'flag_new_account','2+ chargebacks':'flag_chargebacks_2plus',
           'International + card not present':'flag_intl_cnp','Amount > 3x customer average':'flag_amount_3x'}
for rule_name,col in rule_cols.items():
    x[col]=rules[rule_name].astype(int)
x['Rules_Triggered']=x[list(rule_cols.values())].sum(axis=1)
x['Risk_Level']=pd.cut(x['Rules_Triggered'],[-1,0,1,2,10],labels=['Low','Medium','High','Critical']).astype(str)

risk=x.groupby('Rules_Triggered').agg(Transactions=('is_fraud','size'),Frauds=('is_fraud','sum'),Amount_Lost=('fraud_amount_lost','sum')).reset_index()
risk['Fraud_Rate_%']=(risk['Frauds']/risk['Transactions']*100).round(2)
risk

In [ ]:
plt.figure(figsize=(8,5))
bars=plt.bar(risk['Rules_Triggered'].astype(str),risk['Fraud_Rate_%'],color='red',alpha=0.8)
plt.bar_label(bars,fmt='%.1f')
plt.xlabel('Number of rules triggered',size=10)
plt.ylabel('Fraud Rate %',size=10)
plt.title('Fraud Rate Rises with Each Extra Warning Sign',size=14)
plt.tight_layout()
plt.show()

In [ ]:
# policy test: review every transaction that triggers at least N rules
rows=[]
for n in [1,2,3,4]:
    flag=x['Rules_Triggered']>=n
    caught=int(x.loc[flag,'is_fraud'].sum())
    fa=int((flag&(x['is_fraud']==0)).sum())
    saved=x.loc[flag,'fraud_amount_lost'].sum()
    rows.append({'Review_if_rules>=':n,'Flagged':int(flag.sum()),'Frauds_Caught':caught,
                 'Recall_%':round(caught/total_fraud*100,1),'Precision_%':round(caught/flag.sum()*100,1),
                 'Money_Saved':round(saved),'Net_Benefit_cost5':round(saved-fa*5),'Net_Benefit_cost10':round(saved-fa*10)})
policy=pd.DataFrame(rows)
policy

**Finding 6**
- **Evidence:** at 5 per false alert, "Amount > 3x customer average" has the highest net benefit of any single rule (115,525) and stays positive at 10 per alert (44,535); "High-risk country" also stays positive (32,356 at 10). Broad rules such as "International + card not present" (193,645 at cost 1) lose money at 10 per alert (-113,390), and so does the combined four-rule filter (-71,839). The risk score does better than any single rule: the fraud rate rises from 0.33% with no rules triggered to 5.6% with two and over 14% with three or more. Reviewing transactions that trigger 2 or more rules catches 60.6% of fraud cases and 222,941 of the 279,842 lost (about 80%), with a net benefit of 144,341 at 5 per alert and 65,741 at 10. At 10 per alert, the stricter "3 or more" policy is best (83,620).
- **Insight:** a rule's value depends on the cost of a false alert, and stacking warning signs beats using one broad rule.
- **Recommendation:** review transactions that trigger 2 or more rules (3 or more if false alerts are expensive), and measure the real cost of a false alert before adding broader rules.

## Final findings and recommendations

| # | Business question | Key finding | Recommendation |
|---|---|---|---|
| 1 | How big is the problem? | 2.05% of transactions are fraud (2,055 of 100,000), 279,842 lost | Target controls, do not check everything |
| 2 | Which fraud type costs most? | Card-Not-Present is 53.8% of losses | Strengthen online authentication first |
| 3 | Where does it concentrate? | Four countries run at 10% to 13%; online 3.18% vs in-store 0.80%; no chip or PIN about 3x riskier | Step-up checks for online, international and the four countries |
| 4 | When does it happen? | 1am to 4am at 7.5% to 11.3% but only 7% of fraud; no weekend or monthly pattern | Overnight monitoring only as a secondary control |
| 5 | Who is at risk? | New accounts 4.85%; 2+ chargebacks 4.3% to 4.8%; age, email, state show no signal | Extra checks for new accounts and repeat chargebacks |
| 6 | Which rules pay off? | "Amount > 3x average" and "High-risk country" are profitable at every alert cost; a risk score (reviewing 2+ triggered rules) nets 144,341 vs 115,525 for the best single rule | Review transactions with 2 or more triggers; 3 or more if alerts are costly |

### Executive summary
Fraud affects about 2 in every 100 transactions and cost 279,842. More than half of the loss comes from card-not-present payments, and fraud is concentrated in online payments, four countries, new accounts and customers with repeated chargebacks. Simple rules can recover most of the loss, but only if the bank counts the cost of false alerts.

### Limitations
- Savings are estimates: they assume a rule prevents the full loss and each false alert costs a flat 5.
- Rules were designed and tested on the same data, so results on future data would likely be lower.
- The data appears synthetic (it has dates through December 2026), so the project shows method, not real bank performance.
- Some groups (overnight hours, individual countries and states) have small transaction counts.

### Next steps
- Measure the real cost of a false alert.
- Test the rules on a later period.
- Review the rules monthly.

## Save the tables for Power BI
A star schema: one fact table of transactions, plus customer and date lookup tables, and a rule summary table.

In [ ]:
os.makedirs('powerbi_data',exist_ok=True)
pb=x.copy()
pb['date']=pb['transaction_datetime'].dt.normalize()
pb['time_of_day']=pd.cut(pb['transaction_hour'],[-1,4,11,17,23],labels=['Overnight (0-4)','Morning (5-11)','Afternoon (12-17)','Evening (18-23)']).astype(str)
pb['amount_band']=pd.cut(pb['transaction_amount'],[-1,25,100,250,500,1000,1e9],labels=['<25','25-100','100-250','250-500','500-1000','1000+']).astype(str)
pb['card_present_label']=pb['card_present'].map({1:'Card Present',0:'Card Not Present'})
pb['is_fraud_label']=pb['is_fraud'].map({1:'Fraud',0:'Legitimate'})
pb['Risk_Level_Order']=pb['Risk_Level'].map({'Low':1,'Medium':2,'High':3,'Critical':4})

fact_cols=['transaction_id','customer_id','card_id','merchant_id','date','transaction_datetime','transaction_amount',
 'merchant_category','merchant_country','transaction_hour','time_of_day','transaction_day_of_week','is_weekend',
 'card_present_label','transaction_channel','device_type','browser','international_transaction','high_risk_country',
 'used_chip','used_pin','amount_band','amount_vs_avg_ratio','email_domain','is_fraud','is_fraud_label',
 'fraud_type','fraud_amount_lost']+list(rule_cols.values())+['Rules_Triggered','Risk_Level','Risk_Level_Order']
pb[fact_cols].to_csv('powerbi_data/fact_transactions.csv',index=False)

cust=pb.groupby('customer_id').first()[['customer_age','age_band','customer_gender','customer_state','credit_score',
 'credit_limit','num_cards_held','account_age_days','account_age_band','num_chargebacks_ever']].reset_index()
cust.to_csv('powerbi_data/dim_customer.csv',index=False)

d=pd.DataFrame({'date':pd.date_range(pb['date'].min(),pb['date'].max())})
d['year']=d['date'].dt.year
d['month_num']=d['date'].dt.month
d['month_name']=d['date'].dt.strftime('%b')
d['quarter']='Q'+d['date'].dt.quarter.astype(str)
d['day_name']=d['date'].dt.day_name()
d['day_num']=d['date'].dt.dayofweek+1
d.to_csv('powerbi_data/dim_date.csv',index=False)

# rule summary (the cost of a false alert is applied in Power BI with a slider)
rs=[]
for name,mask in rules.items():
    if name=='Combined (4 rules)': continue
    rs.append({'Rule':name,'Flagged':int(mask.sum()),'Frauds_Caught':int(x.loc[mask,'is_fraud'].sum()),
               'False_Alerts':int((mask&(x['is_fraud']==0)).sum()),'Money_Saved':round(x.loc[mask,'fraud_amount_lost'].sum(),2)})
pd.DataFrame(rs).to_csv('powerbi_data/rule_summary.csv',index=False)
print(sorted(os.listdir('powerbi_data')))